# Granger test: Scale_1 and corrected cultural production

In [1]:
import sys

import duckdb
import numpy as np
import pandas as pd
from scipy.stats import combine_pvalues
from statsmodels.tsa.stattools import grangercausalitytests

sys.path.insert(0, '..')
from config import ROOT, SESHAT_DB

UNSEEN_SPECIES = ROOT / 'data/processed/unseen_species.csv'
POLITIES_XLSX = ROOT / 'external/SeshatDatasetAnalysis/datasets/polities.xlsx'
LAG = 2
SIGNIFICANCE = 0.05

pd.options.display.float_format = '{:.3f}'.format

## Data

In [2]:
unseen_species = pd.read_csv(UNSEEN_SPECIES)
print(len(unseen_species), 'home region and century rows')
unseen_species

273 home region and century rows


,home_seshat_region,century,f0,f0_min,f0_max,f1,f2,f>2
0,Afghanistan,1000,38.056,6.487,180.208,8,2,8
1,Afghanistan,1100,13.061,2.156,62.341,4,0,1
2,Afghanistan,1200,7.421,1.231,35.701,2,0,1
3,Afghanistan,1700,31.518,5.990,145.419,4,0,0
4,Anatolia-Caucasus,-500,20.154,2.262,139.630,3,0,0
...,...,...,...,...,...,...,...,...
268,Western Europe,1300,786.764,615.086,1003.724,158,16,47
269,Western Europe,1400,1155.694,924.650,1442.737,175,22,83
270,Western Europe,1500,5849.841,4837.371,7077.850,682,45,128
271,Western Europe,1600,24251.349,20520.233,28705.987,2203,90,288


In [3]:
scale_values = pd.read_excel(POLITIES_XLSX, sheet_name='SCV_Imputed', usecols=['PolityName', 'Year', 'Scale_1']).dropna()
print(len(scale_values), 'Scale_1 values for', scale_values['PolityName'].nunique(), 'polities')
scale_values

1152 Scale_1 values for 576 polities


,PolityName,Year,Scale_1
0,cn_qing_dyn_1,1644,8.070
1,cn_qing_dyn_1,1796,8.070
2,cn_qing_dyn_2,1796,8.188
3,cn_qing_dyn_2,1912,8.253
4,mx_basin_of_mexico_1,-6000,3.379
...,...,...,...
1475,gb_british_emp_2,1968,8.812
1496,de_empire_1,919,6.959
1497,de_empire_1,1125,6.959
1498,de_east_francia,842,6.338


In [4]:
con = duckdb.connect(str(SESHAT_DB), read_only=True)
polity_regions = con.sql('SELECT polity_new_id AS seshat_id, home_seshat_region FROM polity').df()
con.close()
polity_regions

,seshat_id,home_seshat_region
0,cn_qing_dyn_1,North China
1,cn_qing_dyn_2,North China
2,mx_basin_of_mexico_1,Mexico
3,mx_basin_of_mexico_2,Mexico
4,mx_basin_of_mexico_3,Mexico
...,...,...
859,fr_france_modern_2,Western Europe
860,eg_egypt_modern_1,Northeast Africa
861,eg_egypt_modern_2,Northeast Africa
862,sy_syria_modern,Levant


## Panel by home region and century

In [5]:
polity_scale = scale_values.groupby('PolityName').agg(
    start_year=('Year', 'min'),
    end_year=('Year', 'max'),
    complexity=('Scale_1', 'mean'),
).reset_index()
polity_scale = polity_scale.rename(columns={'PolityName': 'seshat_id'}).merge(polity_regions, on='seshat_id')
polity_scale

,seshat_id,start_year,end_year,complexity,home_seshat_region
0,Early Niynginya,1650,1897,4.312,East Africa
1,af_durrani_emp,1747,1826,6.393,Afghanistan
2,af_ghur_principality,1025,1215,6.097,Afghanistan
3,af_greco_bactrian_k,-256,-125,6.435,Afghanistan
4,af_hephthalite_emp,408,561,7.059,Afghanistan
...,...,...,...,...,...
571,ye_ziyad_dyn,822,1037,5.958,Arabia
572,zi_great_zimbabwe,1270,1550,5.263,Southern Africa
573,zi_mutapa,1450,1880,5.388,Southern Africa
574,zi_torwa_rozvi,1494,1850,5.614,Southern Africa


In [6]:
centuries = pd.DataFrame({'century': range(-500, 1800, 100)})
alive = polity_scale.merge(centuries, how='cross')
alive = alive[(alive['start_year'] <= alive['century'] + 99) & (alive['end_year'] >= alive['century'])]
alive

,seshat_id,start_year,end_year,complexity,home_seshat_region,century
21,Early Niynginya,1650,1897,4.312,East Africa,1600
22,Early Niynginya,1650,1897,4.312,East Africa,1700
45,af_durrani_emp,1747,1826,6.393,Afghanistan,1700
61,af_ghur_principality,1025,1215,6.097,Afghanistan,1000
62,af_ghur_principality,1025,1215,6.097,Afghanistan,1100
...,...,...,...,...,...,...
13238,zi_toutswe,700,1250,4.125,Southern Africa,800
13239,zi_toutswe,700,1250,4.125,Southern Africa,900
13240,zi_toutswe,700,1250,4.125,Southern Africa,1000
13241,zi_toutswe,700,1250,4.125,Southern Africa,1100


In [7]:
complexity = alive.groupby(['home_seshat_region', 'century'])['complexity'].mean().reset_index()
complexity

,home_seshat_region,century,complexity
0,Afghanistan,-300,6.435
1,Afghanistan,-200,5.711
2,Afghanistan,-100,4.987
3,Afghanistan,0,6.016
4,Afghanistan,100,7.045
...,...,...,...
613,Western North America,1100,5.111
614,Western North America,1200,5.111
615,Western North America,1300,5.111
616,Western North America,1400,5.111


In [8]:
production = unseen_species[['home_seshat_region', 'century']].copy()
production['cultural_production'] = unseen_species['f0'] + unseen_species['f1'] + unseen_species['f2'] + unseen_species['f>2']

panel = production.merge(complexity, on=['home_seshat_region', 'century'])
panel['log_cultural_production'] = np.log(panel['cultural_production'])
print(len(panel), 'home region and century rows |', panel['home_seshat_region'].nunique(), 'home regions')
panel

243 home region and century rows | 24 home regions


,home_seshat_region,century,cultural_production,complexity,log_cultural_production
0,Afghanistan,1000,56.056,6.097,4.026
1,Afghanistan,1100,18.061,6.097,2.894
2,Afghanistan,1200,10.421,6.097,2.344
3,Afghanistan,1700,35.518,6.393,3.570
4,Anatolia-Caucasus,-300,60.204,6.135,4.098
...,...,...,...,...,...
238,Western Europe,1300,1007.764,6.527,6.915
239,Western Europe,1400,1435.694,6.643,7.269
240,Western Europe,1500,6704.841,6.892,8.811
241,Western Europe,1600,26832.349,7.230,10.197


## Test on one region: North China

In [9]:
region = panel[panel['home_seshat_region'] == 'North China'].sort_values('century')
region

,home_seshat_region,century,cultural_production,complexity,log_cultural_production
102,North China,-500,25.954,5.994,3.256
103,North China,-400,30.772,6.443,3.427
104,North China,-300,32.566,6.662,3.483
105,North China,-200,98.888,7.318,4.594
106,North China,-100,72.760,7.318,4.287
107,North China,0,45.032,7.353,3.807
108,North China,100,39.083,7.389,3.666
109,North China,200,127.152,7.208,4.845
110,North China,300,45.994,6.930,3.829
111,North China,400,34.616,6.736,3.544


In [10]:
changes = region[['log_cultural_production', 'complexity']].diff().dropna()
changes

,log_cultural_production,complexity
103,0.170,0.449
104,0.057,0.219
105,1.111,0.656
106,-0.307,0.000
107,-0.480,0.036
108,-0.142,0.036
109,1.180,-0.182
110,-1.017,-0.278
111,-0.284,-0.194
112,0.117,0.747


In [11]:
result = grangercausalitytests(changes[['log_cultural_production', 'complexity']], maxlag=[LAG])
f_statistic, p_value, df_denominator, df_numerator = result[LAG][0]['ssr_ftest']
print('complexity -> log_cultural_production | F =', round(f_statistic, 3), '| p =', round(p_value, 3))

result = grangercausalitytests(changes[['complexity', 'log_cultural_production']], maxlag=[LAG])
f_statistic, p_value, df_denominator, df_numerator = result[LAG][0]['ssr_ftest']
print('log_cultural_production -> complexity | F =', round(f_statistic, 3), '| p =', round(p_value, 3))

complexity -> log_cultural_production | F = 0.477 | p = 0.63
log_cultural_production -> complexity | F = 0.443 | p = 0.65


## By home region

In [12]:
rows = []
for region_name, region in panel.groupby('home_seshat_region'):
    region = region.sort_values('century')
    changes = region[['log_cultural_production', 'complexity']].diff().dropna()
    if len(changes) < 3 * LAG + 3:
        continue

    result = grangercausalitytests(changes[['log_cultural_production', 'complexity']], maxlag=[LAG])
    f_statistic, p_value, _, _ = result[LAG][0]['ssr_ftest']
    rows.append({'home_seshat_region': region_name, 'test': 'complexity -> log_cultural_production',
                 'observations': len(changes) - LAG, 'f_statistic': f_statistic, 'p_value': p_value})

    result = grangercausalitytests(changes[['complexity', 'log_cultural_production']], maxlag=[LAG])
    f_statistic, p_value, _, _ = result[LAG][0]['ssr_ftest']
    rows.append({'home_seshat_region': region_name, 'test': 'log_cultural_production -> complexity',
                 'observations': len(changes) - LAG, 'f_statistic': f_statistic, 'p_value': p_value})

by_region = pd.DataFrame(rows).sort_values(['test', 'p_value']).reset_index(drop=True)
by_region['significant'] = by_region['p_value'] < SIGNIFICANCE
with pd.option_context('display.max_rows', None):
    display(by_region)

,home_seshat_region,test,observations,f_statistic,p_value,significant
0,Maghreb,complexity -> log_cultural_production,7,66.101,0.015,True
1,Western Europe,complexity -> log_cultural_production,11,7.445,0.024,True
2,Southern Europe,complexity -> log_cultural_production,18,3.572,0.058,False
3,Northeast Asia,complexity -> log_cultural_production,9,2.025,0.247,False
4,Northeast Africa,complexity -> log_cultural_production,8,1.957,0.286,False
5,Anatolia-Caucasus,complexity -> log_cultural_production,17,1.202,0.334,False
6,North China,complexity -> log_cultural_production,20,0.477,0.630,False
7,Mongolia,complexity -> log_cultural_production,8,0.167,0.854,False
8,Iran,complexity -> log_cultural_production,17,0.138,0.872,False
9,Southern India,complexity -> log_cultural_production,11,0.122,0.887,False


## All home regions

In [13]:
pooled = []
for test, results in by_region.groupby('test'):
    statistic, p_value = combine_pvalues(results['p_value'], method='fisher')
    pooled.append({'test': test, 'regions': len(results), 'observations': results['observations'].sum(),
                   'fisher_statistic': statistic, 'p_value': p_value})
pooled = pd.DataFrame(pooled)
pooled['significant'] = pooled['p_value'] < SIGNIFICANCE
pooled

,test,regions,observations,fisher_statistic,p_value,significant
0,complexity -> log_cultural_production,11,141,30.955,0.097,False
1,log_cultural_production -> complexity,11,141,23.857,0.355,False


## Fixed-effects regression

One observation is a home region in a century. Century and home region fixed effects; standard errors clustered by home region.

In [14]:
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

model = smf.ols('log_cultural_production ~ complexity + C(century) + C(home_seshat_region)', panel).fit(
    cov_type='cluster', cov_kwds={'groups': pd.factorize(panel['home_seshat_region'])[0]})

In [15]:
table = summary_col(
    [model],
    model_names=['log cultural production'],
    stars=True,
    float_format='%.3f',
    regressor_order=['complexity'],
    drop_omitted=True,
    info_dict={
        'Century fixed effects': lambda m: 'Yes',
        'Home region fixed effects': lambda m: 'Yes',
        'Observations': lambda m: f'{int(m.nobs)}',
        'Home regions': lambda m: f'{panel["home_seshat_region"].nunique()}',
    },
)
table.add_title('Scale_1 and corrected cultural production (standard errors clustered by home region)')
print(table)

Scale_1 and corrected cultural production (standard errors clustered by home region)
                          log cultural production
-------------------------------------------------
complexity                1.152***               
                          (0.246)                
R-squared                 0.772                  
R-squared Adj.            0.718                  
Century fixed effects     Yes                    
Home region fixed effects Yes                    
Observations              243                    
Home regions              24                     
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01
